# BME Lab 114 — Morphology-Elasticity Relationship of Trabecular Bone

**Author:** Simone Poncioni, MSB  
**Date:** Spring Semester 2026

---

## Notebook 2: Mesh Generation & Simulation

In the previous notebook, we produced a clean binary segmentation of the trabecular bone. We now have a 3D mask of bone vs. void — but to run a Finite Element simulation, we need to convert this image into a **mesh**: a structured assembly of elements, each carrying material properties and boundary conditions.

This notebook covers the full pipeline from segmented image to simulation generation:

1. Loading and downsampling the segmented image
2. Isolating the largest connected bone component
3. Generating a voxel-based FE mesh
4. Visualising the mesh
5. Writing the simulation input file (`.inp`)

### 0. Imports

In [21]:
from pathlib import Path
import sys
import numpy as np
import itk
import vtk
from itkwidgets import view
import scipy.ndimage as ndi

parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))

import ciclope

from utils.img_io import read_image_metadata, read_image
from utils.viewer_utils import display_image_cursor

### 1. Load and Downsample the Segmented Image

We load the binary segmentation produced in Notebook 1. Since the full-resolution image would generate millions of elements — making the simulation prohibitively slow — we **downsample** by a factor `DS` before meshing.

> **Note:** Downsampling reduces computation time but also reduces spatial resolution. Think about what trade-off is acceptable for your analysis.

In [22]:
# img_path = Path("../../00_DATA/00_test/derived/Sample1_segmented.mha")
img_path = Path("../../00_DATA/group01/A1/derived/C0004351_segmented.mha")

# downsample image factor DS
DS = 6

bone, _ = read_image(img_path)
bone_ds = itk.shrink_image_filter(bone, shrink_factors=[DS, DS, DS])
spacing = np.ones(3) * 0.0343994 #TODO: update voxel size
spacing = spacing * DS
bone_arr = itk.array_from_image(bone_ds)

### 2. Visualise the Downsampled Image

Inspect the downsampled bone mask. Check that the trabecular structure is still clearly visible and that no major features have been lost due to downsampling.

In [23]:
display_image_cursor(bone_arr, 'Downsampled image')

interactive(children=(IntSlider(value=36, description='Slice:', max=71), Output()), _dom_classes=('widget-inte…

### 3. Isolate Largest Component and Generate the Voxel Mesh

After downsampling, small disconnected fragments may appear. We retain only the **largest connected component** to ensure the mesh represents a single, mechanically meaningful bone structure.

Each bone voxel is then converted into a hexahedral finite element (a small cube), producing an **unstructured grid** that faithfully follows the trabecular geometry. The mesh is saved as a `.vtk` file for inspection and further processing.

In [24]:
# Keep only largest component
labeled, num = ndi.label(bone_arr)
print("Number of components found:", num)

if num == 0:
    raise ValueError("No connected components found in bone_arr.")

sizes = ndi.sum(bone_arr, labeled, range(1, num + 1))
largest_label = np.argmax(sizes) + 1
bone_arr = (labeled == largest_label).astype(np.uint8)


# Generate voxel FE mesh
print("\n--- GENERATING MESH ---")

mesh = ciclope.core.voxelFE.vol2ugrid(bone_arr, spacing, verbose=True)

# Write VTK
output_path = img_path.with_suffix(".vtk")
mesh.write(output_path, binary=True)
print(f"VTK written to: {output_path}")

INFO:root:Calculating cell array


Number of components found: 79

--- GENERATING MESH ---


INFO:root:Detecting node coordinates and boundary nodes
INFO:root:Generated the following mesh with 443840 nodes and 130214 elements:
INFO:root:<meshio mesh object>
  Number of points: 443840
  Number of cells:
    hexahedron: 130214
  Point sets: NODES_Y1, NODES_Y0, NODES_X0, NODES_X1, NODES_Z1, NODES_Z0, NODES_X0Y0Z0, NODES_X0Y0Z1
  Cell sets: CELLS_Y1, CELLS_Y0, CELLS_X0, CELLS_X1, CELLS_Z1, CELLS_Z0
  Cell data: GV


VTK written to: ../../00_DATA/group01/A1/derived/C0004351_segmented.vtk


### 4. Visualise the Mesh

We load the generated `.vtk` file and visualise the mesh geometry. At this stage you should see the full 3D trabecular structure discretised into hexahedral elements.

In [25]:
reader = vtk.vtkUnstructuredGridReader()
reader.SetFileName(output_path)
reader.Update()
grid = reader.GetOutput()
view(geometries=grid)

2026-03-03 13:59:08.200 (1121.207s) [    7F3482306540]      vtkDataReader.cxx:1966   ERR| vtkUnstructuredGridReader (0x6a04070): Unsupported data type: vtktypeuint8


Viewer(geometries=[{'vtkClass': 'vtkPolyData', 'points': {'vtkClass': 'vtkPoints', 'name': '_points', 'numberO…

### 5. Generate the Simulation Input File

The mesh alone is not enough to run a simulation — we also need to define **boundary conditions**, **material properties**, and **loading**. These are specified in an Abaqus-compatible `.inp` file, generated here from a template using `ciclope`.

The template encodes the physics of the problem: a uniaxial compression test with fixed nodes at the bottom face and a prescribed displacement at the top.

In [26]:
mesh_path     = img_path.with_suffix(".vtk")
template_sim  = Path("/home/bmelab/bmelabs/2026/testing/MSB_BMELab/114-MorphologyElasticityTrabecularBone/muFE/01_CODE/utils/template_sim.inp")
inp_path      = mesh_path.with_suffix(".inp")

ciclope.core.voxelFE.mesh2voxelfe(mesh, template_sim, inp_path, verbose=True)

INFO:root:Found cell_data: GV. cell_data range: 1 - 1.
INFO:root:Start writing INP file
INFO:root:Writing model nodes to INP file
INFO:root:Writing model elements to INP file
INFO:root:Additional nodes sets generated: ['NODES_Y1', 'NODES_Y0', 'NODES_X0', 'NODES_X1', 'NODES_Z1', 'NODES_Z0', 'NODES_X0Y0Z0', 'NODES_X0Y0Z1']
INFO:root:Additional cell sets generated: ['CELLS_Y1', 'CELLS_Y0', 'CELLS_X0', 'CELLS_X1', 'CELLS_Z1', 'CELLS_Z0']
INFO:root:Reading Abaqus template file /home/bmelab/bmelabs/2026/testing/MSB_BMELab/114-MorphologyElasticityTrabecularBone/muFE/01_CODE/utils/template_sim.inp
INFO:root:Model with 443840 nodes and 130214 elements written to file ../../00_DATA/group01/A1/derived/C0004351_segmented.inp
